# Moduł 3: Pandas — analiza danych i statystyka

Pandas to biblioteka do pracy z **danymi tabelarycznymi** (jak Excel/SQL, ale w Pythonie).

## Spis treści
1. DataFrame i Series
2. Ładowanie danych
3. Eksploracja danych
4. Indeksowanie i filtrowanie
5. Grupowanie i agregacja
6. Brakujące wartości
7. Statystyka opisowa
8. **Zadania do samodzielnego rozwiązania**

---

In [ ]:
import pandas as pd
import numpy as np

## 1. DataFrame i Series

**Series** — jednowymiarowa tablica z indeksem (jak kolumna w tabeli).

**DataFrame** — dwuwymiarowa tabela z nazwanymi kolumnami i indeksem wierszy.

In [ ]:
# Tworzenie Series
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s)
print(s["b"]) # 20

# Tworzenie DataFrame ze słownika
df = pd.DataFrame({
 "imie": ["Anna", "Bartek", "Celina", "Dawid"],
 "wiek": [22, 35, 28, 42],
 "miasto": ["Warszawa", "Kraków", "Warszawa", "Gdańsk"],
 "wynagrodzenie": [5000, 8000, 6500, 9500]
})
print(df)

## 2. Ładowanie danych

Pandas czyta wiele formatów: CSV, Excel, JSON, SQL, HTML...

In [ ]:
# Tworzenie przykładowego zbioru danych (symulacja zamiast czytania pliku)
np.random.seed(42)
n = 200
df_students = pd.DataFrame({
 "id": range(1, n + 1),
 "plec": np.random.choice(["K", "M"], n),
 "wiek": np.random.randint(18, 26, n),
 "kierunek": np.random.choice(["Informatyka", "Matematyka", "Fizyka", "Biologia"], n),
 "ocena_koncowa": np.round(np.random.uniform(2.0, 5.0, n), 1),
 "stypendium": np.random.choice([True, False], n, p=[0.3, 0.7])
})

# Dodajmy brakujące wartości
mask = np.random.random(n) < 0.05
df_students.loc[mask, "ocena_koncowa"] = np.nan

print(df_students.head())

## 3. Eksploracja danych

Zawsze zacznij od poznania swojego zbioru danych!

In [ ]:
# Pierwsze / ostatnie wiersze
print(df_students.head(3))
print(df_students.tail(3))

# Rozmiar i kształt
print(f"Kształt: {df_students.shape}") # (200, 6)
print(f"Kolumny: {df_students.columns.tolist()}")
print(f"Typy danych:\n{df_students.dtypes}")

# Info (podsumowanie typów i braków)
df_students.info()

# Statystyki opisowe
print(df_students.describe()) # tylko kolumny numeryczne

## 4. Indeksowanie i filtrowanie

In [ ]:
# Wybieranie kolumn
print(df_students["imie" if "imie" in df_students.columns else "kierunek"].head()) # Series
print(df_students[["kierunek", "ocena_koncowa"]].head()) # DataFrame (kilka kolumn)

# Filtrowanie (warunki)
informatycy = df_students[df_students["kierunek"] == "Informatyka"]
print(f"Informatycy: {len(informatycy)}")

# Wiele warunków: & (i), | (lub), ~ (nie)
wybitni = df_students[
 (df_students["ocena_koncowa"] >= 4.5) & 
 (df_students["stypendium"] == True)
]
print(f"Wybitni stypendyści: {len(wybitni)}")

# loc (po nazwie) vs iloc (po pozycji)
print(df_students.loc[0:2, ["kierunek", "ocena_koncowa"]]) # wiersze 0-2
print(df_students.iloc[0:2, 3:5]) # wiersze 0-1, kolumny 3-4

## 5. Grupowanie i agregacja

`groupby()` — odpowiednik SQL `GROUP BY`. Grupuje dane i oblicza statystyki per grupa.

In [ ]:
# Średnia ocena na kierunek
print(df_students.groupby("kierunek")["ocena_koncowa"].mean())

# Wiele funkcji naraz
print(df_students.groupby("kierunek")["ocena_koncowa"].agg(["mean", "std", "count"]))

# Grupowanie po wielu kolumnach
print(df_students.groupby(["kierunek", "plec"])["ocena_koncowa"].mean())

# value_counts — ile wystąpień każdej wartości
print(df_students["kierunek"].value_counts())

## 6. Brakujące wartości (NaN)

Dane z rzeczywistego świata prawie ZAWSZE mają braki. Trzeba je obsłużyć!

In [ ]:
# Ile braków w każdej kolumnie?
print(df_students.isna().sum())

# Usuwanie wierszy z brakami
df_clean = df_students.dropna()
print(f"Przed: {len(df_students)}, po usunięciu braków: {len(df_clean)}")

# Wypełnianie braków (imputation)
df_filled = df_students.copy()
srednia_ocena = df_filled["ocena_koncowa"].mean()
df_filled["ocena_koncowa"] = df_filled["ocena_koncowa"].fillna(srednia_ocena)
print(f"Braki po wypełnieniu: {df_filled['ocena_koncowa'].isna().sum()}")

## 7. Statystyka opisowa — kluczowe pojęcia

| Pojęcie | Opis | NumPy/Pandas |
|---------|------|-------------|
| **Średnia** | Suma / n | `mean()` |
| **Mediana** | Wartość środkowa | `median()` |
| **Odchylenie std** | Rozrzut danych | `std()` |
| **Wariancja** | σ² | `var()` |
| **Kwantyle** | Wartości dzielące dane | `quantile([0.25, 0.5, 0.75])` |
| **Korelacja** | Związek liniowy (-1 do 1) | `corr()` |

### Średnia vs mediana
- **Średnia** jest wrażliwa na wartości odstające (outliers)
- **Mediana** jest odporna — lepiej opisuje "typowy" element

In [ ]:
# Przykład: średnia vs mediana
zarobki = pd.Series([3000, 3500, 4000, 4500, 100000]) # jeden outlier
print(f"Średnia: {zarobki.mean():.0f}") # ~23000 — zawyżona!
print(f"Mediana: {zarobki.median():.0f}") # 4000 — realistyczna

# Kwantyle
oceny = df_students["ocena_koncowa"].dropna()
print(f"\nKwartyle (Q1, Q2, Q3):")
print(oceny.quantile([0.25, 0.5, 0.75]))

# Korelacja
print(f"\nMacierz korelacji:")
print(df_students[["wiek", "ocena_koncowa"]].corr())

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Analiza danych studentów (łatwe)

Używając `df_students`:
1. Ile jest studentek (K) i studentów (M)?
2. Jaki jest średni wiek studentów na każdym kierunku?
3. Który kierunek ma najwyższą średnią ocenę?

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: Filtrowanie złożone (średnie)

Znajdź studentów, którzy:
- Studiują Informatykę LUB Matematykę
- Mają ocenę ≥ 4.0
- NIE mają stypendium

Wypisz ich dane posortowane malejąco po ocenie.

In [ ]:
# Zadanie 2
# TWÓJ KOD TUTAJ

### Zadanie 3: Tabela krzyżowa (średnie)

Stwórz tabelę krzyżową pokazującą **średnią ocenę** w podziale na kierunek (wiersze) i płeć (kolumny).

Użyj `pd.pivot_table()`.

In [ ]:
# Zadanie 3
# TWÓJ KOD TUTAJ

### Zadanie 4: Nowa kolumna obliczeniowa (średnie)

Dodaj nową kolumnę `kategoria` do `df_students`:
- `"wybitny"` jeśli ocena ≥ 4.5
- `"dobry"` jeśli 3.5 ≤ ocena < 4.5
- `"przeciętny"` jeśli ocena < 3.5

Użyj `np.select()` lub `pd.cut()` lub `apply()`.

In [ ]:
# Zadanie 4
# TWÓJ KOD TUTAJ

### Zadanie 5: Analiza pogodowa — budowanie DataFrame (trudne)

Stwórz DataFrame z danymi pogodowymi dla 365 dni:
- `data` — daty od 01.01.2024 do 31.12.2024 (`pd.date_range`)
- `temperatura` — losowe wartości: zima (-5 to 5°C), wiosna (5-20), lato (15-35), jesień (0-15)
- `opady_mm` — losowe 0-30 mm

Następnie oblicz:
1. Średnią temperaturę na miesiąc
2. Najcieplejszy i najzimniejszy dzień
3. Liczbę dni z opadami > 20 mm per kwartał

In [ ]:
# Zadanie 5
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 6: Laczenie DataFrame (srednie)

Stworz dwa DataFrame:
- `df_oceny`: kolumny `id_ucznia`, `przedmiot`, `ocena` (20 wierszy)
- `df_uczniowie`: kolumny `id_ucznia`, `imie`, `klasa` (10 uczniow)

Wykonaj:
1. `merge` (inner join) po `id_ucznia`
2. Oblicz srednia ocene per klasa
3. Znajdz ucznia z najwyzsza srednia
4. Stworz pivot table: wiersze = uczniowie, kolumny = przedmioty, wartosci = oceny

In [ ]:
# Zadanie 6: Laczenie DataFrame
import pandas as pd
import numpy as np
np.random.seed(42)
# TWOJ KOD TUTAJ

### Zadanie 7: Rolling i window functions (trudne)

Wygeneruj szereg czasowy (365 dni):
- Wartosc = trend liniowy + sezonowoscsinusoidalna + szum

Oblicz:
1. Srednia kroczaca (rolling mean) z oknem 7 i 30 dni
2. Odchylenie standardowe kroczace (rolling std) z oknem 30 dni
3. Roznice (diff) — zmiana dzien do dnia
4. Narysuj wykres z oryginalnymi danymi i obiema srednimi kroczacymi

In [ ]:
# Zadanie 7: Rolling i window functions
import pandas as pd
import numpy as np
np.random.seed(42)
# TWOJ KOD TUTAJ